# PoC step 4: reverse mappers, end-to-end run, and fidelity checks
Applies the v1.1 patch, runs every test, and reproduces all PoC results with one command.

**Before you start:** copy `reverse_mappers.py`, `test_reverse.py`, `run_all.py`, `patch_v1_1.py`, `README.md`, and `requirements.txt` into `bw-interop-poc/src` (README and requirements can sit one level up in `bw-interop-poc`). `MatchedPoolReference.csv` must be in `data`.

In [1]:
# Cell 1: mount Drive and install dependencies
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/bw-interop-poc'
!pip -q install "pydantic>=2" pyyaml pytest

Mounted at /content/drive


In [2]:
# Cell 2: apply the v1.1 patch (safe to run twice; every line should say patched or already patched)
!python {BASE}/src/patch_v1_1.py --src {BASE}/src

schema.sql           patched: CHECK (charge_bearer IN ('DEBT','CRED','SHARE  ->  CHECK (charge_bearer IN ('DEBT','CRED','SHARE
schema.sql           patched: SLEV normalised to SHARED  ->  SLEV kept as SERVICE_LEVEL (no information lo
loader.py            patched: "SLEV": "SHARED"  ->  "SLEV": "SERVICE_LEVEL"
make_poc_pairs.py    patched: "reference": f"REF-{n:06d}"  ->  "reference": iso["rmt"]
DONE


In [3]:
# Cell 3: run all tests (expect 58 passed: 36 + 14 + 8)
!cd {BASE}/src && python -m pytest -q -p no:cacheprovider

..........................................................               [100%]
58 passed in 10.00s


In [4]:
# Cell 4: one command reproduces everything (expect ALL CHECKS PASSED)
!python {BASE}/src/run_all.py --base {BASE} --pool {BASE}/data/MatchedPoolReference.csv


=== 1. Generate synthetic pairs ===
pacs.008 records: 402  Orange records: 394  manifest rows: 408
expected_outcome
MATCHED       352
DISCREPANT     24
REJECTED       20
UNMATCHED      12
anomaly_type
(clean)             336
DUPLICATE_E2E        16
AMOUNT_MISMATCH      12
FAILED_STATUS        12
LAG_OUTLIER           8
MISSING_CURRENCY      6
ORPHAN_OM             6
ORPHAN_ISO            6
UNKNOWN_BIC           6

=== 2. Validate and load ===
PACS008  rows_in= 402 mapped= 382 rejected= 20 reasons={'MISSING_CURRENCY': 6, 'DUPLICATE_KEY': 8, 'UNKNOWN_BIC': 6}
ORANGE   rows_in= 394 mapped= 394 rejected=  0 reasons={}
transactions             776
transaction_source_ids   3474
parties                  782
status_reports           776
exceptions               20
load_audit               2

=== 3. Reconcile, post ledger, score ===
Reconciliation outcomes: {'MATCHED': 352, 'DISCREPANT': 24, 'UNMATCHED': 24}
Journal entries: 704 | unbalanced transactions: 0
Accuracy vs manifest: 408/408 = 100.

In [5]:
# Cell 5: one payment shown three ways: bank record, translated wallet request, actual wallet leg
import json
sample = json.load(open(f'{BASE}/data/out_samples.json'))
for title, body in sample.items():
    print(f'--- {title} ---')
    print(json.dumps(body, indent=2))

--- bank_record ---
{
  "msg_id": "MSG-BW-20260801-POC",
  "instr_id": "INSTR-BW-00001",
  "end_to_end_id": "E2E-BW-20260807-0001",
  "tx_id": "TX-BW-00001",
  "uetr": "daf61a26-146d-4f31-bc37-7a4c4a15544d",
  "clr_sys_ref": "CLR-BW-00001",
  "amount": "18521.71",
  "currency": "BWP",
  "settlement_date": "2026-08-08",
  "charge_bearer": "CRED",
  "debtor_name": "DBTR-231457",
  "debtor_account": "1155104816",
  "debtor_bic": "MOCKBWBRXXX",
  "creditor_name": "CDTR-160276",
  "creditor_account": "26779399039",
  "creditor_bic": "MOCKBWOMXXX",
  "remittance_info": "Invoice 2026-0001"
}
--- translated_to_wallet ---
{
  "order_id": "E2E-BW-20260807-0001",
  "reference": "Invoice 2026-0001",
  "amount": {
    "value": 18521.71,
    "unit": "BWP"
  },
  "customer": {
    "id": "26779399039",
    "idType": "MSISDN"
  },
  "partner": {
    "id": "BARODA",
    "idType": "CODE"
  },
  "status": "ACCEPTED"
}
--- actual_wallet_leg ---
{
  "transactionId": "OM-20260807-00001",
  "order_id": "E2E-B

In [6]:
# Cell 6: field-level fidelity of the round trip (percentage of records where the field came back identical)
import pandas as pd
fid = pd.read_csv(f'{BASE}/data/out_fidelity.csv', dtype=str)
fid['same'] = fid['same'] == 'True'
table = fid.groupby(['source', 'field'])['same'].mean().mul(100).round(1).unstack(0)
print(table.fillna('').to_string())

source           ORANGE PACS008
field                          
amount            100.0   100.0
charge_bearer             100.0
clr_sys_ref               100.0
createdAt         100.0        
creditor_account          100.0
creditor_bic              100.0
creditor_name             100.0
currency                  100.0
customer_id       100.0        
customer_idType   100.0        
debtor_account            100.0
debtor_bic                100.0
debtor_name               100.0
end_to_end_id             100.0
instr_id                  100.0
msg_id                    100.0
order_id          100.0        
partner_id        100.0        
partner_idType    100.0        
pay_token         100.0        
reference         100.0        
remittance_info           100.0
settlement_date           100.0
status            100.0        
transactionId     100.0        
tx_id                     100.0
uetr                      100.0
unit              100.0        


## Done when
- Cell 2 says `DONE`.
- Cell 3 ends with `58 passed`.
- Cell 4 prints `ALL CHECKS PASSED` and 776/776 for the round trip.
- Cell 5 shows the translated wallet request matching the actual wallet leg on order_id, reference, amount, customer, and partner.
- Cell 6 shows 100.0 for every field.

Paste the printed output of cells 2, 3, and 4 into the decision log. After the patch the earlier datasets are regenerated, so any numbers you copied before keep the same counts but the wallet `reference` text changes.